# FIFA 24 Player Stats Analysis
Authors: Erik Argueta & Logan Brenner

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from plotnine import *

# Get The Data

In [ ]:
fifa_df = pd.read_csv("Material/fifa_stats.csv")

In [ ]:
fifa_df.head(10)

In [ ]:
fifa_df.shape

# Exploring the Data (Exploratory Data Analysis - EDA)

In [ ]:
fifa_df.dtypes

In [ ]:
(
    fifa_df
    .duplicated().any()
)

In [ ]:
fifa_df.describe().T

## Categorizing Variables

In [ ]:
fifa_cleaning = (
    fifa_df
)

### Remove Unnecessary Columns

In [ ]:
fifa_cleaning.drop(columns=['gk_positioning', 'marking', 'gk_diving', 'gk_handling', 'gk_kicking', 'gk_reflexes'], inplace=True)

In [ ]:
fifa_cleaning.shape

### Exploring Category: club

In [ ]:
fifa_cleaning["club"].value_counts()

In [ ]:
fifa_cleaning['club'] = fifa_cleaning['club'].str.strip()

In [ ]:
(
    fifa_cleaning
    .query('`club` == "River Plate"')
)

## Normalizing value

In [ ]:
fifa_cleaning["log_market_value"] = np.log(fifa_df['value'])

In [ ]:
(
    fifa_cleaning
    .filter(['log_market_value'])
    .sort_values(by='log_market_value', ascending=False)
)

In [ ]:
fifa_cleaning = (
    fifa_df.astype({'country': 'category', 'club': 'category'})
)

# Data Visualization

In [ ]:
fifa_cleaning.dtypes

In [ ]:
g = sns.PairGrid(fifa_cleaning)
g.map(sns.scatterplot)
plt.show()

In [ ]:
corr_matrix = fifa_cleaning.select_dtypes('number').corr()

plt.figure(figsize=(25,20))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap for FIFA Dataset")
plt.show()

### Strongest Correlations:

log_market_value x ['reactions', 'composure', 'short_pass']

In [ ]:
(
    fifa_cleaning
    .filter(['ID','club', 'reactions', 'composure', 'short_pass', 'value', 'log_market_value'])
)

In [ ]:
(
    ggplot(fifa_cleaning)
    + aes(x="reactions", y="value")
    + geom_point()
    + labs(
        title="Market Value by Reaction Score for all FIFA Players"
    )
)

In [ ]:
(
    ggplot(fifa_cleaning)
    + aes(x="composure", y="value")
    + geom_point()
    + labs(
        title="Market Value by Composure Score for All FIFA Players"
    )
)

In [ ]:
(
    ggplot(fifa_cleaning)
    + aes(x="short_pass", y="value")
    + geom_point()
    + labs(
        title="Market Value by Short Pass for all FIFA Players"
    )
)

# Final Remarks
Correlation of variables in predicting `value` or `log_market_value`
* reactions (0.74)
* composure (0.61)
* short_pass (0.51)

**NOTE:** There are 1136 NaN `value` and `log_market_value`. Not sure what to do with them...

# Preparing the Data for Linear Regression

In [ ]:
#Drop rows where there is no value
import numpy as np

df_model = fifa_cleaning.dropna(subset=['value']).copy()

df_model['log_market_value'] = np.log1p(df_model['value'])

df_model[['value', 'log_market_value']].head()

In [ ]:
#Some values were really low like like 1500 Id assume it should be 1.5 mil so I do this math to adjust 
df_model['value_corrected'] = np.where(
    df_model['value'] < 10000,
    df_model['value'] * 1000,
    df_model['value']
)

df_model['log_market_value_corrected'] = np.log1p(df_model['value_corrected'])

In [ ]:
features = [
    'reactions',
    'composure',
    'short_pass',
    'ball_control',
    'vision',
    'long_pass',
    'shot_power',
    'finishing',
    'age'
]

X = df_model[features]
y = df_model['log_market_value_corrected']

X = X.fillna(X.median())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

## Evaluation Metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R² Score:", r2)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Actual Log Market Value")
plt.ylabel("Predicted Log Market Value")
plt.title("Actual vs Predicted Log Market Value")

plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])

plt.show()

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 6))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0)
plt.xlabel("Predicted Log Market Value")
plt.ylabel("Residuals")
plt.title("Residual Plot")

plt.show()

In [ ]:
coefficients = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_
}).sort_values(by='Coefficient', ascending=False)

coefficients

In [ ]:
corr_matrix = df_model.select_dtypes('number').corr()

plt.figure(figsize=(25,20))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap for FIFA Dataset")
plt.show()

# Fine Tuning the Model
Following the textbook, we implemented GridSearch and RandomForestRegressor to fine-tune the model and define the best model.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

param_grid = [
    { 'n_estimators': [3, 10, 30], 'max_features': [2, 4, 6, 8] },
    { 'bootstrap': [False], 'n_estimators': [3, 10], 'max_features': [2, 3, 4] }
]

forest_reg = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV( estimator=forest_reg, param_grid=param_grid, cv=5,
    scoring='neg_mean_squared_error', return_train_score=True, n_jobs=-1 )

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print("Best Parameters:", grid_search.best_params_)

# Analyzing the Finalized Model Against Evaluation Metrics and Test Set

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R² Score:", r2)

| Metric | Initial Model | Fine-Tuned Model | Interpretation |
|---|---|---|---|
| MAE | 0.5139 | 0.5139 | Average prediction error remained about the same |
| RMSE | 0.6836 | 0.4962 | Large prediction errors were reduced significantly |
| R² Score | 0.7643 | 0.8758 | Model explains more variance in the data after tuning |

In [ ]:
df = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R² Score'] * 2,
    'Score': [0.5139, 0.6836, 0.7643,
              0.3667, 0.4962, 0.8758],
    'Model': ['Initial Model'] * 3 + ['Fine-Tuned Model'] * 3
})

(
    ggplot(df, aes(x='Metric', y='Score', fill='Model'))
    + geom_bar(stat='identity', position='dodge')
    + labs(
        title='Model Performance Comparison',
        x='Metric',
        y='Score'
    )
)

In [ ]:
scatter_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred
})

(
    ggplot(scatter_df, aes(x='Actual', y='Predicted'))
    + geom_point()
    + labs(
        title='Actual vs Predicted Values of Fine-Tuned Data',
        x='Actual Values',
        y='Predicted Values'
    )
    + theme_minimal()
)

In [ ]:
residual_df = pd.DataFrame({
    'Predicted': y_pred,
    'Residuals': y_test - y_pred
})

(
    ggplot(residual_df, aes(x='Predicted', y='Residuals'))
    + geom_point()
    + geom_hline(yintercept=0, linetype='dashed')
    + labs(
        title='Residual Plot of Fine-Tuned Data',
        x='Predicted Values',
        y='Residuals'
    )
    + theme_minimal()
)


# Saving the Model

In [ ]:
import joblib

joblib.dump(model, "fifa_linear_regression_model.joblib")

In [ ]:
joblib.dump(best_model, "fifa_random_forest_model.joblib")